# PPO 0721：7 个实验 / 3 组对照的综合审计

## TL;DR

在严格 critic 对照中，privilege_gru 的 5 个 checkpoint 碰撞数为 16/18/18/14/14，均值 16.0，优于 independent_gru 的 25.6 和 privilege_mlp 的 24.0。

当前证据支持保留 privilege_gru、batch_size=12800；clip=0.15 是现有面板上的首选，clip=0.20 是需要 worker-matched 多 seed 复验的候选。

## Context and decision

分析目标是同时回答 critic 结构、batch size 和 clip range 的取舍，并审计训练稳定性、checkpoint 参数位移、Austin 600 固定面板和 NPZ trace 的可信度。

In [1]:
from pathlib import Path
import pandas as pd
DATA = Path('analysis/ppo_0721_seven_experiments/data')
def load(name): return pd.read_csv(DATA / f'{name}.csv')
control = load('control_audit')
training = load('training_summary')
eval_rollup = load('eval_rollup')
paired = load('paired_comparisons')
actor = load('actor_parameter_deltas')
trace = load('trace_quality')
print(f'Loaded {len(control)} runs, {len(eval_rollup) * 5} checkpoint evals, {trace.expected_trace_files.sum():,.0f} NPZ traces expected.')


Loaded 7 runs, 35 checkpoint evals, 21,000 NPZ traces expected.


## Data quality and control-variable audit

In [2]:
cols=['run','critic','batch_size','clip_range','env_workers','formal_updates','eval_checkpoints','eval_episodes_min','warmup_equals_local_baseline','warmup_equals_clip010','seed_persisted','source_commit_persisted']
print(control[cols].to_string(index=False))


            run          critic  batch_size  clip_range  env_workers  formal_updates  eval_checkpoints  eval_episodes_min  warmup_equals_local_baseline  warmup_equals_clip010  seed_persisted  source_commit_persisted
independent_gru independent_gru       12800        0.15           12              20                 5                600                          True                  False           False                    False
  privilege_mlp  priviledge_mlp       12800        0.15           12              20                 5                600                          True                  False           False                    False
  privilege_gru   privilege_gru       12800        0.15           12              20                 5                600                          True                  False           False                    False
    batch_25600   privilege_gru       25600        0.15           12              20                 5                600               

![Austin checkpoint trajectories](figures/eval_collision_trajectories.png)

三张图分别对应 critic、batch size、clip range。clip=0.15 与另外两档存在 env_workers 混入，因此第三张图是结果参考，不是完全干净的三档因果比较。

## Training metrics and optimization dynamics

In [3]:
cols=['run','warmup_best_validation_loss','ev_last5','collision_ev_last5','rollout_collision_first5','rollout_collision_last5','kl_mean_median','kl_mean_p95','kl_single_minibatch_max','actor_preclip_grad_norm_median','total_training_minutes']
print(training[cols].round(4).to_string(index=False))


            run  warmup_best_validation_loss  ev_last5  collision_ev_last5  rollout_collision_first5  rollout_collision_last5  kl_mean_median  kl_mean_p95  kl_single_minibatch_max  actor_preclip_grad_norm_median  total_training_minutes
independent_gru                       0.1926    0.9106              0.9042                    0.3131                   0.2642          0.0342       0.1580                   3.4958                         37.5046                 53.6750
  privilege_mlp                       0.1832    0.6942              0.6474                    0.3541                   0.2992          0.0480       0.4189                   2.6625                         39.9041                 32.4639
  privilege_gru                       0.2982    0.9123              0.9054                    0.3542                   0.2532          0.0327       0.3156                   3.0944                         43.3518                 56.5674
    batch_25600                       0.2608    0.8718  

![Training diagnostics](figures/training_diagnostics.png)

## Checkpoint parameter movement

In [4]:
final_actor=actor.loc[actor['update']==20, ['run','actor_relative_l2_from_bc','gru_relative_l2_from_bc','head_relative_l2_from_bc','fixed_frontend_delta_l2']]
print(final_actor.round(7).to_string(index=False))


            run  actor_relative_l2_from_bc  gru_relative_l2_from_bc  head_relative_l2_from_bc  fixed_frontend_delta_l2
independent_gru                   0.000538                 0.000252                  0.002595                      0.0
  privilege_mlp                   0.000549                 0.000250                  0.002667                      0.0
  privilege_gru                   0.000543                 0.000237                  0.002669                      0.0
    batch_25600                   0.000434                 0.000180                  0.002156                      0.0
    batch_51200                   0.000374                 0.000152                  0.001860                      0.0
       clip_010                   0.000459                 0.000201                  0.002255                      0.0
       clip_020                   0.000680                 0.000283                  0.003373                      0.0


![Actor parameter displacement](figures/actor_parameter_displacement.png)

## Austin 600 outcomes and paired evidence

In [5]:
cols=['run','collision_sequence','collision_mean','best_update','best_collision_count','final_collision_count','final_overtake_count','final_follow_count']
print(eval_rollup[cols].to_string(index=False))


            run collision_sequence  collision_mean  best_update  best_collision_count  final_collision_count  final_overtake_count  final_follow_count
independent_gru     19/19/27/29/34            25.6            1                    19                     34                   329                 237
  privilege_mlp     24/25/25/21/25            24.0           15                    21                     25                   360                 215
  privilege_gru     16/18/18/14/14            16.0           15                    14                     14                   349                 237
    batch_25600     18/22/21/22/16            19.8           20                    16                     16                   348                 236
    batch_51200     22/17/15/17/21            18.4           10                    15                     21                   342                 237
       clip_010     28/23/17/28/20            23.2           10                    17         

In [6]:
final_pairs=paired.loc[paired['update']==20, ['group','control_strength','left_run','right_run','delta_left_minus_right','left_only_collisions','right_only_collisions','mcnemar_exact_p','cluster_bootstrap_delta_ci_low','cluster_bootstrap_delta_ci_high']]
print(final_pairs.round(4).to_string(index=False))


 group    control_strength        left_run     right_run  delta_left_minus_right  left_only_collisions  right_only_collisions  mcnemar_exact_p  cluster_bootstrap_delta_ci_low  cluster_bootstrap_delta_ci_high
critic              strict independent_gru privilege_gru                      20                    25                      5           0.0003                            11.0                             30.0
critic              strict   privilege_mlp privilege_gru                      11                    17                      6           0.0347                             2.0                             20.0
 batch              strict     batch_25600 privilege_gru                       2                     6                      4           0.7539                            -4.0                              8.0
 batch              strict     batch_51200 privilege_gru                       7                    13                      6           0.1671                          

## Scenario concentration

In [7]:
scenario=load('scenario_risk')
print(scenario.head(15)[['episode_key','collision_count','checkpoint_count','collision_frequency','risk_band']].to_string(index=False))
print('universal=', int((scenario.collision_frequency==1).sum()), '>=80%=', int((scenario.collision_frequency>=.8).sum()), 'ever=', int((scenario.collision_count>0).sum()))


         episode_key  collision_count  checkpoint_count  collision_frequency      risk_band
  ol0_e213_o221_s0.7               35                35             1.000000      universal
ol1_e1497_o1512_s0.5               35                35             1.000000      universal
  ol2_e727_o745_s0.5               35                35             1.000000      universal
ol1_e1368_o1383_s0.6               30                35             0.857143 near_universal
  ol2_e641_o661_s0.7               27                35             0.771429       frequent
  ol0_e256_o262_s0.8               21                35             0.600000       frequent
  ol0_e898_o902_s0.7               21                35             0.600000       frequent
  ol2_e727_o745_s0.7               21                35             0.600000       frequent
   ol2_e85_o106_s0.8               21                35             0.600000       frequent
ol0_e1539_o1533_s0.7               20                35             0.571429    

![Scenario slice risk](figures/scenario_slice_risk.png)

## Privileged feature and trace health

In [8]:
features=load('feature_health_summary')
print(features[['feature','mean_std','max_exact_low_fraction','max_exact_high_fraction','max_fraction_ge_0_99','normalization_interpretation']].round(4).to_string(index=False))


                   feature  mean_std  max_exact_low_fraction  max_exact_high_fraction  max_fraction_ge_0_99                normalization_interpretation
      cos_relative_heading    0.2775                  0.0000                   0.0209                0.5992        natural cosine concentration near +1
   cos_track_heading_error    0.0687                  0.0000                   0.0023                0.3220        natural cosine concentration near +1
         current_curvature    0.3843                  0.0574                   0.0251                0.0257           expected percentile-tail clipping
                   delta_s    0.3865                  0.0000                   0.0936                0.0949                                     healthy
            ego_slip_angle    0.0880                  0.0000                   0.0001                0.0001                                     healthy
                 ego_speed    0.1585                  0.0000                   0.0000   

In [9]:
print(trace[['run','update','missing_trace_files','trace_length_mismatches','steering_clip_frame_rate','json_collision_count','trace_collision_any_count','json_collisions_missing_terminal_trace_frame']].to_string(index=False))


            run  update  missing_trace_files  trace_length_mismatches  steering_clip_frame_rate  json_collision_count  trace_collision_any_count  json_collisions_missing_terminal_trace_frame
independent_gru       1                    0                        0                  0.000615                    19                          0                                            19
independent_gru       5                    0                        0                  0.000579                    19                          0                                            19
independent_gru      10                    0                        0                  0.000509                    27                          0                                            27
independent_gru      15                    0                        0                  0.000431                    29                          0                                            29
independent_gru      20                    0 

## Takeaways and limitations

- critic 与 batch 两组控制成立；clip=0.10 vs 0.20 控制成立，但它们与 clip=0.15 的比较混入 env_workers/执行环境。
- 单 seed、固定 Austin 面板和 checkpoint 选优会高估可泛化优势；逐场景配对能减少面板噪声，但不能替代多 seed 与 holdout。
- NPZ 缺 terminal post-step frame，碰撞标签必须以 results_multi.json 为准。
- 下一步只需对 privilege_gru 的 clip=0.15/0.20 做同 worker、同源码哈希、2–3 seed 的窄对照，并加入 target-KL early stop。